In [1]:
# ==========================================
import gc
import os
import warnings
from functools import reduce
from itertools import combinations
from typing import Any, Dict, List, Optional, Tuple, Union
# ==========================================
import dask
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import kurtosis, skew
# ==========================================
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
# ==========================================
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from cartopy.feature import ShapelyFeature
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter # For coordinate tick labels
from shapely.ops import unary_union
# ==========================================

In [31]:
raw_data_folder = "./data/raw/"
kumasi_lat = 6.68848
kumasi_lon = -1.62443

In [4]:
def standardize_SEAS5(ds: xr.Dataset, variable_name: Optional[str] = None) -> xr.Dataset:
    """
    Standardize coordinate names for SEAS5 NetCDF files and compute validity times.
    Optionally, it can standardize units for a specific variable, managing both
    monthly rates and daily accumulated variables.

    This function renames standard coordinates (latitude, longitude, ensemble member, initialization time)
    to a common convention. It automatically detects whether the dataset is daily or monthly based on
    available coordinates.

    For **both monthly and daily datasets**, it additionally calculates/assigns:
    - `valid_time`: A 2D coordinate (init_time, lead_time) representing the actual validity date.
    - `target_month`: The specific month (1-12) derived from the validity time.
    
    Variable unit standardization:
    - If `variable_name` is "tprate" (monthly), it converts the values from m/s to mm/day.
    - If `variable_name` is "tp" (daily), it de-accumulates the precipitation (tp_t2 - tp_t1) 
      and converts the values from meters (m) to millimeters (mm).

    Parameters
    ----------
    ds : xr.Dataset
        Input xarray Dataset containing SEAS5 NetCDF data (from Copernicus CDS).
        Expected to contain coordinates like 'forecast_reference_time' and either
        'forecastMonth' (monthly) or 'forecast_period' (daily).
    variable_name : str, optional
        The name of the variable to standardize units for ("tprate" or "tp").

    Returns
    -------
    xr.Dataset
        Dataset with standardized coordinate names (lat, lon, init_time, lead_time, ensemble_member).
        It will also include `valid_time` and `target_month` coordinates, and potentially 
        converted variable units.
    """
    # Rename common coords
    ds = ds.rename({
        "latitude": "lat",
        "longitude": "lon",
        "number": "ensemble_member",
        "forecast_reference_time": "init_time",
    })

    # Flag to track dataset temporal resolution
    is_daily = False

    if "forecast_period" in ds.coords:
        is_daily = True
        # It's a daily dataset
        ds = ds.rename({
            "forecast_period": "lead_time",
        })
        
        # cfgrib sometimes loads valid_time as a data variable instead of a coordinate.
        if "valid_time" in ds.data_vars:
            ds = ds.set_coords("valid_time")
        # Compute vectorially (datetime64 + timedelta64) if missing
        elif "valid_time" not in ds.coords:
            ds.coords['valid_time'] = ds.init_time + ds.lead_time

        # 'target_month' created from 'valid_time'
        ds.coords['target_month'] = ds.valid_time.dt.month

    else:
        # It's a monthly dataset
        ds = ds.rename({
            "forecastMonth": "lead_time",
        })
        
        # New coordinate system for monthly data
        # valid_time = init_time + lead_time (in months)
        inits = pd.to_datetime(ds.init_time.values)
        leads = ds.lead_time.values
        valid_times = []
        for init in inits:
            row = [(init + pd.DateOffset(months=int(lead))) for lead in leads]
            valid_times.append(row)
            
        # Convert to numpy datetime64 for compatibility with xarray
        valid_times_np = np.array(valid_times, dtype='datetime64[ns]')
        
        # Assign new coordinate 'valid_time' to the dataset
        ds = ds.assign_coords(
            valid_time=(('init_time', 'lead_time'), valid_times_np)
        )
        # 'target_month' created from 'valid_time'
        ds.coords['target_month'] = ds.valid_time.dt.month

    # Standardization of units for specific variables
    if variable_name is not None and variable_name in ds:
        
        if not is_daily and variable_name == "tprate":
            # Monthly conversion: from m/s to mm/day
            ds[variable_name] = ds[variable_name] * 1000 * 86400
            
            if 'GRIB_units' in ds[variable_name].attrs:
                ds[variable_name].attrs['GRIB_units'] = 'mm/day'
                
        elif is_daily and variable_name == "tp":
            # Daily conversion: from accumulated meters (m) to daily millimeters (mm)
            first_day = ds[variable_name].isel(lead_time=slice(0, 1))
            following_days = ds[variable_name].diff(dim='lead_time')
            ds[variable_name] = xr.concat([first_day, following_days], dim='lead_time') * 1000
            ds[variable_name].attrs['units'] = 'mm'
            if 'GRIB_units' in ds[variable_name].attrs:
                ds[variable_name].attrs['GRIB_units'] = 'mm'

    return ds


def crop_nc_to_region(ds : xr.Dataset, lat_min: float, lat_max: float, lon_min: float, lon_max: float) -> xr.Dataset:
    """
    Crop an xarray Dataset to a specified lat/lon bounding box.
    Latitudes and longitudes are in degrees. 
    Columns 'lat' and 'lon' are assumed.

    Parameters
    ----------
    ds : xr.Dataset
        Input xarray Dataset to be cropped.
    lat_min : float
        Minimum latitude of the bounding box.
    lat_max : float
        Maximum latitude of the bounding box.
    lon_min : float
        Minimum longitude of the bounding box.
    lon_max : float
        Maximum longitude of the bounding box.

    Returns
    -------
    xr.Dataset
        Cropped Dataset.
    """
    return ds.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max))



def merge_SEAS5_datasets(datasets: List[xr.Dataset]) -> xr.Dataset:
    """
    Concatenate a list of SEAS5 datasets along the 'init_time' dimension.

    This function merges datasets downloaded in separate chunks (e.g., by 
    different initialization months) that share the same spatial domain, 
    ensemble members, and lead times. It automatically sorts the resulting 
    dataset chronologically to ensure a continuous time series.

    Parameters
    ----------
    datasets : list of xarray.Dataset
        A list of xarray Datasets to be merged. All datasets are expected 
        to have been processed by `standardize_SEAS5` and must include 
        the 'init_time' coordinate.

    Returns
    -------
    xarray.Dataset
        A single, continuous Dataset concatenated and sorted along the 
        'init_time' dimension.

    Raises
    ------
    ValueError
        If the input list `datasets` is empty.
    KeyError
        If the 'init_time' coordinate is missing from the first dataset 
        in the list.
    """
    if not datasets:
        raise ValueError("The input list of datasets is empty. Please provide at least one dataset.")

    # Check for 'init_time' to prevent errors during concatenation
    if 'init_time' not in datasets[0].coords:
        raise KeyError("'init_time' coordinate not found. Ensure `standardize_SEAS5` was run.")

    # Concatenate along 'init_time'. 
    merged_ds = xr.concat(datasets, dim='init_time')
    merged_ds = merged_ds.sortby('init_time')

    return merged_ds


def add_coords_to_daily_SEAS5(ds: xr.Dataset) -> xr.Dataset:
    """
    Transforms a SEAS5 dataset by stacking temporal dimensions and calculating monthly lead times.

    This function reshapes the dataset from a 2D time structure (`init_time`, `lead_time`)
    into a flat 1D time structure (`forecast_time`). It replaces the newly stacked dimension
    with the actual validity time (`valid_time`) and calculates the lead time in months.

    Parameters
    ----------
    ds : xarray.Dataset
        The input SEAS5 dataset. Expected to have `init_time` and `lead_time` as
        dimensions, and `valid_time` as a 2D coordinate.

    Returns
    -------
    xarray.Dataset
        The transformed dataset with a single `forecast_time` dimension (which represents
        the validity time), and a new `monthly_lead_time`.

    Notes
    -----
    The `monthly_lead_time` is calculated as the integer number of months between
    the initialization time (`init_time`) and the validity time (`forecast_time`).
    """
    ds = ds.stack(forecast_time=("init_time", "lead_time"))
    ds = ds.set_index(forecast_time="valid_time")
    ds["forecast_time"].attrs = {
        "standard_name": "valid_time",
        "long_name": "validity time of the forecast",
        "description": "Exact datetime the forecast is valid for"
    }


    if "target_month" in ds.coords:
        ds["target_month"].attrs = {
            "long_name": "month of the forecast",
            "units": "month"
        }

    init_year = ds["init_time"].dt.year
    init_month = ds["init_time"].dt.month
    valid_year = ds["forecast_time"].dt.year
    valid_month = ds["forecast_time"].dt.month
    monthly_lead_time = (valid_year - init_year) * 12 + (valid_month - init_month)
    ds = ds.assign_coords(monthly_lead_time=monthly_lead_time)
    ds["monthly_lead_time"].attrs = {
        "standard_name": "forecast_period",
        "long_name": "lead time in months",
        "units": "months",
        "description": "Integer number of months between initialization and validity time"
    }
    
    return ds



In [5]:
# variable_name = "msl"
# mslp_list = []

# suffixes = np.arange(1992,2022+1,1).tolist()

# suffixes = [str(suffix) for suffix in suffixes]
# coords_to_remove = [
#     'target_month', 
#     'init_time', 
#     'lead_time', 
#     'monthly_lead_time'
# ]

# # suffixes = [1992,1993]
# for suffix in suffixes:
#     print(f"Processing {suffix}...")
#     #skippo mesi non esistenti e produco un warning
#     try:
#         ds_block = xr.open_dataset(f'{raw_data_folder}/SEAS5_mslp_{suffix}.nc')
#         ds_block = standardize_SEAS5(ds_block, 
#                                     variable_name=variable_name)
#         ds_block = add_coords_to_daily_SEAS5(ds_block)
#         ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.month == 6))
#         ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.year > 1992))
#         # ds_block = ds_block.sortby('forecast_time')

#         ds_block = ds_block.drop_vars(coords_to_remove, errors='ignore')
#         mslp_list.append(ds_block)
#         del ds_block
#         gc.collect()
#     except FileNotFoundError:
#         warnings.warn(f"File for {suffix} not found. Skipping this block.")
#         continue

# mslp = xr.concat(mslp_list, dim='forecast_time')
# mslp = mslp.sortby('forecast_time')

# id = np.arange(len(mslp['forecast_time']))
# mslp = mslp.assign_coords(forecast_time=id)
# mslp = mslp.rename({'forecast_time': 'id'})

In [ ]:
variable_name = "msl"
mslp_list = []

suffixes = np.arange(1992,2022+1,1).tolist()
suffixes = [str(suffix) for suffix in suffixes]

coords_to_remove = [
    'target_month', 
    'lead_time', 
    'monthly_lead_time'
]

for suffix in suffixes:
    print(f"Processing {suffix}...")
    try:
        # Assicurati che raw_data_folder sia definito nel tuo ambiente
        ds_block = xr.open_dataset(f'{raw_data_folder}/SEAS5_mslp_{suffix}.nc')
        ds_block = standardize_SEAS5(ds_block, variable_name=variable_name)
        ds_block = add_coords_to_daily_SEAS5(ds_block)
        ds_block = ds_block.isel(ensemble_member=slice(0, 25))
        ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.month == 6))
        ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.year > 1992))

        ds_block = ds_block.drop_vars(coords_to_remove, errors='ignore')
        mslp_list.append(ds_block)
        del ds_block
        gc.collect()
    except FileNotFoundError:
        warnings.warn(f"File for {suffix} not found. Skipping this block.")
        continue

# 1. Concatenazione di tutti i pacchetti caricati
mslp = xr.concat(mslp_list, dim='forecast_time')

# 2. Fusione delle dimensioni 'forecast_time' ed 'ensemble_member'
mslp_stacked = mslp.stack(sample=['forecast_time', 'ensemble_member'])

# 3. Ordinamento gerarchico
mslp_sorted = mslp_stacked.sortby(['init_time', 'forecast_time', 'ensemble_member'])

# 4. Conversione da MultiIndex a coordinate piatte
mslp_flat = mslp_sorted.reset_index('sample')

# 5. Rinomina in 'id' e assegnazione della sequenza numerica (0, 1, 2, ...)
mslp_final = mslp_flat.rename({'sample': 'id'})
mslp_final['id'] = np.arange(len(mslp_final['id']))


Processing 1992...
Processing 1993...
Processing 1994...
Processing 1995...
Processing 1996...
Processing 1997...
Processing 1998...
Processing 1999...
Processing 2000...
Processing 2001...
Processing 2002...
Processing 2003...
Processing 2004...
Processing 2005...
Processing 2006...
Processing 2007...
Processing 2008...
Processing 2009...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...


In [16]:
mslp_final

<xarray.Dataset> Size: 1GB
Dimensions:          (lat: 41, lon: 51, id: 159575)
Coordinates:
  * lat              (lat) float64 328B 30.0 29.0 28.0 27.0 ... -8.0 -9.0 -10.0
  * lon              (lon) float64 408B -30.0 -29.0 -28.0 ... 18.0 19.0 20.0
  * id               (id) int64 1MB 0 1 2 3 4 ... 159571 159572 159573 159574
    init_time        (id) datetime64[ns] 1MB 1992-11-01 ... 2022-06-01
    forecast_time    (id) datetime64[ns] 1MB 1993-06-01 ... 2022-06-30
    ensemble_member  (id) int64 1MB 0 1 2 3 4 5 6 7 ... 17 18 19 20 21 22 23 24
Data variables:
    msl              (lat, lon, id) float32 1GB 1.028e+05 ... 1.011e+05
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-28T07:10 GRIB to CDM+CF via cfgrib-0.9.1...

In [17]:
# 6. Pulizia finale: ora che l'ordinamento è fatto, scartiamo 'init_time'
mslp_final = mslp_final.drop_vars(['init_time', 'forecast_time', 'ensemble_member'], errors='ignore')

In [20]:
mslp_final.to_netcdf('./data/processed/mslp.nc')

In [ ]:
variable_name = "tp"
precip_list = []

suffixes_precip = ["1992_1996", "1997_2001", "2002_2006", "2007_2011", "2012_2016", "2017_2021", "2022_2022"]

coords_to_remove = [
    'target_month', 
    'lead_time', 
    'monthly_lead_time'
]

for suffix in suffixes_precip:
    print(f"Processing {suffix}...")
    try:
        # Assicurati che raw_data_folder sia definito nel tuo ambiente
        ds_block = xr.open_dataset(f'{raw_data_folder}/SEAS5_tp_{suffix}.nc')
        ds_block = standardize_SEAS5(ds_block, variable_name=variable_name)
        ds_block = add_coords_to_daily_SEAS5(ds_block)
        ds_block = ds_block.isel(ensemble_member=slice(0, 25))
        ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.month == 6))
        ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.year > 1992))

        ds_block = ds_block.drop_vars(coords_to_remove, errors='ignore')
        precip_list.append(ds_block)
        del ds_block
        gc.collect()
    except FileNotFoundError:
        warnings.warn(f"File for {suffix} not found. Skipping this block.")
        continue

# 1. Concatenazione di tutti i pacchetti caricati
precip = xr.concat(precip_list, dim='forecast_time')

# 2. Fusione delle dimensioni 'forecast_time' ed 'ensemble_member'
precip_stacked = precip.stack(sample=['forecast_time', 'ensemble_member'])

# 3. Ordinamento gerarchico
precip_sorted = precip_stacked.sortby(['init_time', 'forecast_time', 'ensemble_member'])

# 4. Conversione da MultiIndex a coordinate piatte
precip_flat = precip_sorted.reset_index('sample')

# 5. Rinomina in 'id' e assegnazione della sequenza numerica (0, 1, 2, ...)
precip_final = precip_flat.rename({'sample': 'id'})
precip_final['id'] = np.arange(len(precip_final['id']))


Processing 1992_1996...
Processing 1997_2001...
Processing 2002_2006...
Processing 2007_2011...
Processing 2012_2016...
Processing 2017_2021...
Processing 2022_2022...


In [21]:
# 6. Pulizia finale: ora che l'ordinamento è fatto, scartiamo 'init_time'
precip_final = precip_final.drop_vars(['init_time', 'forecast_time', 'ensemble_member'], errors='ignore')

In [ ]:
precip_final.to_netcdf('./data/processed/precip.nc')

In [23]:
precip_final

<xarray.Dataset> Size: 1GB
Dimensions:  (lat: 41, lon: 51, id: 159575)
Coordinates:
  * lat      (lat) float64 328B 30.0 29.0 28.0 27.0 ... -7.0 -8.0 -9.0 -10.0
  * lon      (lon) float64 408B -30.0 -29.0 -28.0 -27.0 ... 17.0 18.0 19.0 20.0
  * id       (id) int64 1MB 0 1 2 3 4 5 ... 159570 159571 159572 159573 159574
Data variables:
    tp       (lat, lon, id) float32 1GB 0.0 0.06104 0.3662 3.418 ... 0.0 0.0 0.0
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-28T01:00 GRIB to CDM+CF via cfgrib-0.9.1...

In [24]:
mslp_final

<xarray.Dataset> Size: 1GB
Dimensions:  (lat: 41, lon: 51, id: 159575)
Coordinates:
  * lat      (lat) float64 328B 30.0 29.0 28.0 27.0 ... -7.0 -8.0 -9.0 -10.0
  * lon      (lon) float64 408B -30.0 -29.0 -28.0 -27.0 ... 17.0 18.0 19.0 20.0
  * id       (id) int64 1MB 0 1 2 3 4 5 ... 159570 159571 159572 159573 159574
Data variables:
    msl      (lat, lon, id) float32 1GB 1.028e+05 1.025e+05 ... 1.011e+05
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-28T07:10 GRIB to CDM+CF via cfgrib-0.9.1...

In [25]:
variable_name = "z500"
z500_list = []

suffixes_z500 = ["1992_1993", "1994_1995", "1996_1997", "1998_1999", "2000_2001", "2002_2003", "2004_2005", "2006_2007", "2008_2009", "2010_2011", "2012_2013", 
                 "2014_2015", "2016_2017", "2018_2019", "2020_2021", "2022_2022"]
coords_to_remove = [
    'target_month', 
    'lead_time', 
    'monthly_lead_time'
]

for suffix in suffixes_z500:
    print(f"Processing {suffix}...")
    try:
        # Assicurati che raw_data_folder sia definito nel tuo ambiente
        ds_block = xr.open_dataset(f'{raw_data_folder}/SEAS5_z500_{suffix}.nc')
        ds_block = standardize_SEAS5(ds_block, variable_name=variable_name)
        ds_block = add_coords_to_daily_SEAS5(ds_block)
        ds_block = ds_block.isel(ensemble_member=slice(0, 25))
        ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.month == 6))
        ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.year > 1992))

        ds_block = ds_block.drop_vars(coords_to_remove, errors='ignore')
        z500_list.append(ds_block)
        del ds_block
        gc.collect()
    except FileNotFoundError:
        warnings.warn(f"File for {suffix} not found. Skipping this block.")
        continue

# 1. Concatenazione di tutti i pacchetti caricati
z500 = xr.concat(z500_list, dim='forecast_time')

# 2. Fusione delle dimensioni 'forecast_time' ed 'ensemble_member'
z500_stacked = z500.stack(sample=['forecast_time', 'ensemble_member'])

# 3. Ordinamento gerarchico
z500_sorted = z500_stacked.sortby(['init_time', 'forecast_time', 'ensemble_member'])

# 4. Conversione da MultiIndex a coordinate piatte
z500_flat = z500_sorted.reset_index('sample')

# 5. Rinomina in 'id' e assegnazione della sequenza numerica (0, 1, 2, ...)
z500_final = z500_flat.rename({'sample': 'id'})
z500_final['id'] = np.arange(len(z500_final['id']))

# 6. Pulizia finale: ora che l'ordinamento è fatto, scartiamo 'init_time'
z500_final = z500_final.drop_vars(['init_time', 'forecast_time', 'ensemble_member'], errors='ignore')


Processing 1992_1993...
Processing 1994_1995...
Processing 1996_1997...
Processing 1998_1999...
Processing 2000_2001...
Processing 2002_2003...
Processing 2004_2005...
Processing 2006_2007...
Processing 2008_2009...
Processing 2010_2011...
Processing 2012_2013...
Processing 2014_2015...
Processing 2016_2017...
Processing 2018_2019...
Processing 2020_2021...
Processing 2022_2022...


In [26]:
z500_final.to_netcdf('./data/processed/z500.nc')

In [27]:
variable_name = "q850"
q850_list = []

suffixes_q850 = ["1992_1993", "1994_1995", "1996_1997", "1998_1999", "2000_2001", "2002_2003", "2004_2005", "2006_2007", "2008_2009", "2010_2011", "2012_2013", 
                 "2014_2015", "2016_2017", "2018_2019", "2020_2021", "2022_2022"]
coords_to_remove = [
    'target_month', 
    'lead_time', 
    'monthly_lead_time'
]

for suffix in suffixes_q850:
    print(f"Processing {suffix}...")
    try:
        # Assicurati che raw_data_folder sia definito nel tuo ambiente
        ds_block = xr.open_dataset(f'{raw_data_folder}/SEAS5_q850_{suffix}.nc')
        ds_block = standardize_SEAS5(ds_block, variable_name=variable_name)
        ds_block = add_coords_to_daily_SEAS5(ds_block)
        ds_block = ds_block.isel(ensemble_member=slice(0, 25))
        ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.month == 6))
        ds_block = ds_block.sel(forecast_time=(ds_block['forecast_time'].dt.year > 1992))

        ds_block = ds_block.drop_vars(coords_to_remove, errors='ignore')
        q850_list.append(ds_block)
        del ds_block
        gc.collect()
    except FileNotFoundError:
        warnings.warn(f"File for {suffix} not found. Skipping this block.")
        continue

# 1. Concatenazione di tutti i pacchetti caricati
q850 = xr.concat(q850_list, dim='forecast_time')

# 2. Fusione delle dimensioni 'forecast_time' ed 'ensemble_member'
q850_stacked = q850.stack(sample=['forecast_time', 'ensemble_member'])

# 3. Ordinamento gerarchico
q850_sorted = q850_stacked.sortby(['init_time', 'forecast_time', 'ensemble_member'])

# 4. Conversione da MultiIndex a coordinate piatte
q850_flat = q850_sorted.reset_index('sample')

# 5. Rinomina in 'id' e assegnazione della sequenza numerica (0, 1, 2, ...)
q850_final = q850_flat.rename({'sample': 'id'})
q850_final['id'] = np.arange(len(q850_final['id']))

# 6. Pulizia finale: ora che l'ordinamento è fatto, scartiamo 'init_time'
q850_final = q850_final.drop_vars(['init_time', 'forecast_time', 'ensemble_member'], errors='ignore')


Processing 1992_1993...
Processing 1994_1995...
Processing 1996_1997...
Processing 1998_1999...
Processing 2000_2001...
Processing 2002_2003...
Processing 2004_2005...
Processing 2006_2007...
Processing 2008_2009...
Processing 2010_2011...
Processing 2012_2013...
Processing 2014_2015...
Processing 2016_2017...
Processing 2018_2019...
Processing 2020_2021...
Processing 2022_2022...


In [28]:
q850_final.to_netcdf('./data/processed/q850.nc')

In [29]:
mslp_reopen = xr.open_dataset('./data/processed/mslp.nc')

In [30]:
mslp_reopen

<xarray.Dataset> Size: 1GB
Dimensions:  (lat: 41, lon: 51, id: 159575)
Coordinates:
  * lat      (lat) float64 328B 30.0 29.0 28.0 27.0 ... -7.0 -8.0 -9.0 -10.0
  * lon      (lon) float64 408B -30.0 -29.0 -28.0 -27.0 ... 17.0 18.0 19.0 20.0
  * id       (id) int64 1MB 0 1 2 3 4 5 ... 159570 159571 159572 159573 159574
Data variables:
    msl      (lat, lon, id) float32 1GB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-28T07:10 GRIB to CDM+CF via cfgrib-0.9.1...

In [32]:
import numpy as np
import pandas as pd
import xarray as xr
from typing import Tuple

def extract_pot_target(
    precip_ds: xr.Dataset, 
    var_name: str, 
    lon_target: float, 
    lat_target: float, 
    percentile_threshold: float = 95.0
) -> Tuple[pd.DataFrame, float]:
    """
    Extracts the time series for a single grid point and applies the Peaks Over 
    Threshold (POT) method to generate a binary target for classification.

    Parameters
    ----------
    precip_ds : xr.Dataset
        The xarray Dataset containing the precipitation data. It is expected to be
        flattened along an 'id' dimension.
    var_name : str
        The name of the precipitation variable within the dataset (e.g., 'tprate').
    lon_target : float
        The longitude coordinate of the target grid point.
    lat_target : float
        The latitude coordinate of the target grid point.
    percentile_threshold : float, optional
        The percentile (0-100) to use as the threshold for determining extremes.
        Default is 95.0.

    Returns
    -------
    Tuple[pd.DataFrame, float]
        A tuple containing:
        - df_target (pd.DataFrame): A dataframe containing the 'id' and the binary 
          'target_pot' (1 for extreme, 0 otherwise).
        - threshold_value (float): The physical precipitation value corresponding 
          to the computed threshold.
    """
    
    # 1. Select the grid point using the nearest neighbor method
    print(f"Extracting data for coordinates: Lon {lon_target}, Lat {lat_target}...")
    tp_point: xr.DataArray = precip_ds[var_name].sel(lon=lon_target, lat=lat_target, method='nearest')
    
    # Ensure data is loaded into memory as a numpy array for faster computation
    tp_values: np.ndarray = tp_point.values
    
    # 2. Calculate the threshold based on the percentile over the entire distribution
    threshold_value: float = float(np.nanpercentile(tp_values, percentile_threshold))
    print(f"Extreme threshold calculated ({percentile_threshold}th percentile): {threshold_value:.4f}")
    
    # 3. Apply POT: 1 if extreme, 0 otherwise (np.where handles NaNs safely)
    is_extreme: np.ndarray = np.where(tp_values > threshold_value, 1, 0)
    
    # 4. Insert into a clean DataFrame ready for merging with PCs
    df_target = pd.DataFrame({
        'id': precip_ds['id'].values,
        'target_pot': is_extreme
    })
    
    return df_target, threshold_value

In [34]:
df_target, threshold = extract_pot_target(
    precip_ds=precip_final, 
    var_name='tp', 
    lon_target=kumasi_lon, 
    lat_target=kumasi_lat, 
    percentile_threshold=95
)

Extracting data for coordinates: Lon -1.62443, Lat 6.68848...
Extreme threshold calculated (95th percentile): 17.8223


In [38]:
df_target

,id,target_pot
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
...,...,...
159570,159570,0
159571,159571,0
159572,159572,0
159573,159573,0


In [37]:
df_target["target_pot"].value_counts()

target_pot
0    151601
1      7974
Name: count, dtype: int64

In [39]:
df_target.to_csv('./data/processed/target_pot.csv', index=False)